# Jarvis Intent Classifier Training

Fine-tune `distilbert-base-uncased` on `data/intents.csv` (train+val) and evaluate on `data/test.csv`.

**Runtime:** Colab T4 GPU. Expected training time: 15–25 minutes.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn huggingface_hub matplotlib seaborn

In [ ]:
import os, json, random, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Adjust this path to where you put your data folder in Drive:
DATA_DIR = '/content/drive/MyDrive/jarvis-intent/data'
OUT_DIR = '/content/drive/MyDrive/jarvis-intent/out'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
intents_df = pd.read_csv(f'{DATA_DIR}/intents.csv')
test_df    = pd.read_csv(f'{DATA_DIR}/test.csv')
print('train+val pool:', len(intents_df))
print('test         :', len(test_df))

LABELS = sorted(intents_df['intent'].unique().tolist())
LABEL2ID = {l:i for i,l in enumerate(LABELS)}
ID2LABEL = {i:l for l,i in LABEL2ID.items()}
print('labels:', LABELS)

In [ ]:
intents_df = intents_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
split = int(0.8 * len(intents_df))
train_df = intents_df.iloc[:split].copy()
val_df   = intents_df.iloc[split:].copy()
for df in (train_df, val_df, test_df):
    df['label'] = df['intent'].map(LABEL2ID)
print('train', len(train_df), 'val', len(val_df), 'test', len(test_df))

In [ ]:
MODEL = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch['query'], truncation=True, max_length=64)

train_ds = Dataset.from_pandas(train_df[['query','label']]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df  [['query','label']]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df [['query','label']]).map(tokenize, batched=True)
collator = DataCollatorWithPadding(tok)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

acc = evaluate.load('accuracy')
f1  = evaluate.load('f1')

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': acc.compute(predictions=preds, references=pred.label_ids)['accuracy'],
        'macro_f1': f1.compute(predictions=preds, references=pred.label_ids, average='macro')['f1'],
    }

In [ ]:
args = TrainingArguments(
    output_dir=f'{OUT_DIR}/checkpoints',
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=20,                  # ~10% of 200 total steps
    fp16=True,                        # ~40% faster on T4, no accuracy loss
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=20,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tok,             # renamed from 'tokenizer' in transformers v5
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
pred = trainer.predict(test_ds)
preds = pred.predictions.argmax(-1)
true  = pred.label_ids
print(f"Test accuracy: {(preds == true).mean():.4f}")
print(classification_report(true, preds, target_names=LABELS, digits=4))

report_dict = classification_report(true, preds, target_names=LABELS, output_dict=True)
pd.DataFrame(report_dict).T.to_csv(f'{OUT_DIR}/per_class_metrics.csv')

cm = confusion_matrix(true, preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, xticklabels=LABELS, yticklabels=LABELS, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (test set)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
import time
model_cpu = trainer.model.to('cpu').eval()
sample_queries = test_df['query'].sample(50, random_state=SEED).tolist()

latencies = []
for q in sample_queries:
    enc = tok(q, return_tensors='pt', truncation=True, max_length=64)
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model_cpu(**enc)
    latencies.append((time.perf_counter() - t0) * 1000)

import numpy as np
print(f'Local CPU latency (ms) — mean: {np.mean(latencies):.1f}  p95: {np.percentile(latencies,95):.1f}')
pd.DataFrame({'cpu_ms': latencies}).describe().to_csv(f'{OUT_DIR}/latency_local.csv')

In [ ]:
from huggingface_hub import login
login()  # paste a write-scope token from huggingface.co/settings/tokens

HUB_REPO = 'YOUR-HF-USERNAME/jarvis-intent-classifier'  # ← change this
trainer.model.push_to_hub(HUB_REPO)
tok.push_to_hub(HUB_REPO)
print('Pushed to', HUB_REPO)

## (Optional) ONNX + INT8 quantization

In [ ]:
!pip install -q optimum[onnxruntime] onnxruntime
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

ort_dir = f'{OUT_DIR}/onnx'
ort_model = ORTModelForSequenceClassification.from_pretrained(HUB_REPO, export=True)
ort_model.save_pretrained(ort_dir)
tok.save_pretrained(ort_dir)

quantizer = ORTQuantizer.from_pretrained(ort_dir)
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer.quantize(save_dir=f'{OUT_DIR}/onnx-int8', quantization_config=qconfig)
print('Quantized model saved to', f'{OUT_DIR}/onnx-int8')